# Construcción reproducible del dataset regional

> **Segundo entregable.** Este notebook construye el dataset regional (unos 40 sitios). En este primer entregable se usó la serie de un sitio (notebook 00b); por eso este notebook se muestra sin ejecutar.

Este notebook construye la base de datos directamente desde las fuentes oficiales, sin archivos locales de partida:

- **NASA POWER** (NASA Langley Research Center): variables agroclimáticas diarias, 1981–2025, para una grilla regional del Caribe colombiano y para el punto de la finca en Sabanagrande.
- **NOAA Climate Prediction Center**: Oceanic Niño Index (ONI).

Decisiones de diseño:

1. **Grilla regional** en los centros de celda de POWER (0,5° × 0,625°), de modo que cada punto sea una celda distinta y no haya casi-duplicados.
2. **Solo puntos sobre tierra** (máscara global de tierra) y con elevación ≤ 400 m, para excluir la Sierra Nevada y la Serranía de Perijá.
3. **La celda regional más cercana a la finca se excluye**: la finca es el sitio de prueba y ningún vecino inmediato debe quedar en entrenamiento.
4. **ONI rezagado** para usar solo el valor ya publicado en cada fecha.
5. Las respuestas de la API se guardan en `data/raw/power/` y el dataset final en `data/processed/`, para que el libro no tenga que volver a descargar.

In [ ]:
# Requiere: pip install global-land-mask
import io, json, sys, time, warnings
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from global_land_mask import globe

sys.path.insert(0, str(Path.cwd().parent))
from src import config as cfg
from src.hidrologia import et0_penman_monteith
warnings.filterwarnings("ignore")

## 1. Fuentes, periodo y variables

In [ ]:
NASA_POWER_BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
NOAA_ONI_URL = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

FECHA_INICIO, FECHA_FIN = cfg.FECHA_INICIO, cfg.FECHA_FIN      # 1981-01-01 a 2025-12-31
PARAMETROS_POWER = cfg.POWER_PARAMS
# El ONI de cada trimestre se publica días después de terminar el trimestre.
# Asignado al día 1 del mes central, 75 días de rezago garantizan que ya estaba publicado.
REZAGO_ONI_DIAS = 75

CRUDO = cfg.DATA_RAW / "power"
CRUDO.mkdir(parents=True, exist_ok=True)
cfg.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## 2. Área de estudio y grilla regional

In [ ]:
LATITUDES = np.arange(cfg.LAT_MIN, cfg.LAT_MAX + 0.001, 0.5)
# -76.875 es el primer centro de celda dentro del límite occidental -76.9°
LONGITUDES = np.arange(-76.875, cfg.LON_MAX + 0.001, 0.625)

puntos_regionales = [(float(la), float(lo)) for la in LATITUDES for lo in LONGITUDES
                     if cfg.LON_MIN <= lo <= cfg.LON_MAX and bool(globe.is_land(la, lo))]

# Exclusión de la celda más cercana a la finca (sitio de prueba)
cercano = min(puntos_regionales, key=lambda p: (p[0] - cfg.LAT_FINCA) ** 2 + (p[1] - cfg.LON_FINCA) ** 2)
puntos_regionales = [p for p in puntos_regionales if p != cercano]
print(f"Celdas de la grilla: {len(LATITUDES) * len(LONGITUDES)}")
print(f"Celda excluida por cercanía a la finca: {cercano}")
print(f"Puntos terrestres candidatos: {len(puntos_regionales)}")

## 3. Conexión con reintentos y descarga con caché

In [ ]:
retry = Retry(total=5, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET"], respect_retry_after_header=True)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers.update({"User-Agent": "JupyterBook-ML-Sabanagrande/1.0"})


def construir_url_power(latitud, longitud):
    consulta = {"parameters": ",".join(PARAMETROS_POWER), "community": "AG",
                "longitude": float(longitud), "latitude": float(latitud),
                "start": FECHA_INICIO, "end": FECHA_FIN, "format": "JSON", "time-standard": "LST"}
    return f"{NASA_POWER_BASE_URL}?{urlencode(consulta)}"


def descargar_punto_power(latitud, longitud, nombre_punto, tipo_punto):
    """Descarga (o lee de la caché) un punto y lo devuelve como DataFrame diario."""
    url = construir_url_power(latitud, longitud)
    cache = CRUDO / f"{nombre_punto}.json"
    if cache.exists():
        contenido = json.loads(cache.read_text(encoding="utf-8"))
    else:
        r = session.get(url, timeout=180)
        r.raise_for_status()
        contenido = r.json()
        cache.write_text(json.dumps(contenido), encoding="utf-8")
        time.sleep(0.25)
    if "properties" not in contenido or "parameter" not in contenido["properties"]:
        raise ValueError(f"NASA POWER no devolvió la estructura esperada para {nombre_punto}.")

    datos = pd.DataFrame(contenido["properties"]["parameter"])
    datos.index.name = "fecha"
    datos = datos.reset_index()
    datos["fecha"] = pd.to_datetime(datos["fecha"].astype(str), format="%Y%m%d")
    cols = [c for c in PARAMETROS_POWER if c in datos.columns]
    datos[cols] = datos[cols].replace(-999, np.nan).astype(float)      # -999 = faltante

    coords = contenido.get("geometry", {}).get("coordinates", [longitud, latitud])
    lon_r = float(coords[0]) if len(coords) >= 1 else float(longitud)
    lat_r = float(coords[1]) if len(coords) >= 2 else float(latitud)
    elev = float(coords[2]) if len(coords) >= 3 and coords[2] is not None else np.nan
    if pd.notna(elev) and elev <= -900:
        elev = np.nan

    for pos, (col, val) in enumerate([("punto", nombre_punto), ("tipo_punto", tipo_punto),
                                      ("lat", lat_r), ("lon", lon_r), ("elevacion_m", elev)], start=1):
        datos.insert(pos, col, val)
    return datos, url, elev


print("Link directo NASA POWER para la finca:")
print(construir_url_power(cfg.LAT_FINCA, cfg.LON_FINCA))

## 4. Descarga de la grilla regional y de la finca

La primera ejecución tarda varios minutos; las siguientes leen la caché.

In [ ]:
tablas, fuentes, descartados = [], [], []
for i, (lat, lon) in enumerate(puntos_regionales, start=1):
    nombre = f"REGIONAL_{i:03d}"
    tabla, url, elev = descargar_punto_power(lat, lon, nombre, "regional")
    if pd.notna(elev) and elev > cfg.ELEVACION_MAX_M:
        descartados.append({"punto": nombre, "lat": lat, "lon": lon, "motivo": f"elevación {elev:.0f} m"})
        continue
    tablas.append(tabla)
    fuentes.append({"punto": nombre, "tipo": "regional", "lat_solicitada": lat, "lon_solicitada": lon,
                    "elevacion_m": elev, "url": url})

tabla_finca, url_finca, elev_finca = descargar_punto_power(cfg.LAT_FINCA, cfg.LON_FINCA,
                                                            cfg.PUNTO_OBJETIVO, "finca_prueba")
tablas.append(tabla_finca)
fuentes.append({"punto": cfg.PUNTO_OBJETIVO, "tipo": "finca_prueba", "lat_solicitada": cfg.LAT_FINCA,
                "lon_solicitada": cfg.LON_FINCA, "elevacion_m": elev_finca, "url": url_finca})

power = (pd.concat(tablas, ignore_index=True).sort_values(["punto", "fecha"])
         .drop_duplicates(["punto", "fecha"]).reset_index(drop=True))
fuentes_power = pd.DataFrame(fuentes)
print("Sitios retenidos:", power.punto.nunique(), "| observaciones:", f"{len(power):,}")
pd.DataFrame(descartados)

## 5. Oceanic Niño Index (ONI)

El ONI es una media móvil de tres meses; cada trimestre se asigna a su mes central y se lleva a escala diaria. Luego se rezaga para que en cada fecha solo esté el valor ya publicado.

In [ ]:
r = session.get(NOAA_ONI_URL, timeout=60)
r.raise_for_status()
oni = pd.read_csv(io.StringIO(r.text), sep=r"\s+", engine="python").rename(
    columns={"SEAS": "temporada_oni", "YR": "anio", "TOTAL": "sst_nino34", "ANOM": "ONI_original"})
mes_central = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
               "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}
oni["mes"] = oni["temporada_oni"].map(mes_central)
oni = oni.dropna(subset=["mes", "anio", "ONI_original"]).astype({"mes": int, "anio": int})
oni["fecha_mes"] = pd.to_datetime({"year": oni["anio"], "month": oni["mes"], "day": 1})

calendario = pd.date_range(oni["fecha_mes"].min(), pd.Timestamp("2025-12-31"), freq="D")
oni_diario = (oni.set_index("fecha_mes")["ONI_original"].sort_index().reindex(calendario).ffill()
              .shift(REZAGO_ONI_DIAS).rename("ONI").rename_axis("fecha").reset_index())
oni_diario.dropna().head()

## 6. Evapotranspiración de referencia (FAO-56)

ET0 no viene en POWER: se calcula con Penman-Monteith a partir de temperatura, humedad, viento, radiación y presión (Allen et al., 1998). Es necesaria para la demanda de agua de los cultivos.

In [ ]:
partes = []
for _, g in power.groupby("punto"):
    g = g.copy()
    g["ET0"] = et0_penman_monteith(g.T2M_MAX, g.T2M_MIN, g.RH2M, g.WS2M, g.ALLSKY_SFC_SW_DWN,
                                   lat=g.lat.iloc[0], dia_juliano=g.fecha.dt.dayofyear,
                                   elevacion_m=np.nan_to_num(g.elevacion_m.iloc[0]), presion_kpa=g.PS).round(3)
    partes.append(g)
power = pd.concat(partes, ignore_index=True)

## 7. Dataset final, control de calidad y guardado

In [ ]:
dataset = power.merge(oni_diario, on="fecha", how="left", validate="many_to_one")
columnas = ["fecha", "punto", "tipo_punto", "lat", "lon", "elevacion_m", *PARAMETROS_POWER, "ET0", "ONI"]
dataset = dataset[columnas].sort_values(["punto", "fecha"]).reset_index(drop=True)

duplicados = int(dataset.duplicated(["punto", "fecha"]).sum())
n_finca = int((dataset["punto"] == cfg.PUNTO_OBJETIVO).sum())
print("Periodo:", dataset.fecha.min().date(), "a", dataset.fecha.max().date())
print("Sitios:", dataset.punto.nunique(), "| observaciones:", f"{len(dataset):,}", "| columnas:", dataset.shape[1])
print("Duplicados sitio-fecha:", duplicados, "| observaciones de la finca:", f"{n_finca:,}")

assert dataset.fecha.min() == pd.Timestamp("1981-01-01")
assert dataset.fecha.max() == pd.Timestamp("2025-12-31")
assert duplicados == 0

dataset.to_csv(cfg.DATA_PROCESSED / cfg.ARCHIVO_DATOS, index=False)
fuentes_power.to_csv(cfg.DATA_PROCESSED / "fuentes_power.csv", index=False)
print("Guardado:", cfg.DATA_PROCESSED / cfg.ARCHIVO_DATOS)

In [ ]:
(dataset.isna().mean().mul(100).round(3).sort_values(ascending=False).rename("% faltantes").to_frame())

In [ ]:
fuentes_power